In [ ]:
import requests
from requests.exceptions import JSONDecodeError
import os
from dotenv import load_dotenv
import datetime 
from datetime import date
from datetime import timedelta
import time
import pandas as pd 
import json


In [ ]:
url = "https://opendata.aemet.es/opendata/api/valores/climatologicos/inventarioestaciones/todasestaciones"

load_dotenv(dotenv_path="../.env")
API_KEY = os.getenv("AEMET_API_KEY").strip().strip("'")
headers = {
    "accept": "application/json",
    "api_key": API_KEY
}

r = requests.get(url, headers=headers)
print(r.status_code)
print(r.text)
datos_respuesta = r.json()
print(datos_respuesta)

data_url = datos_respuesta["datos"]
data_response = requests.get(data_url, headers=headers)
estaciones = data_response.json()
print(estaciones[:5])  # Muestra las primeras 5 estaciones para verificar la respuesta





In [ ]:
def filtrar(campo, valor):
    resultado = [estacion for estacion in estaciones if any(elemento in estacion.get(campo, '') for elemento in valor)]
    return resultado


In [ ]:
provincias_interes = ['ZARAGOZA', 'HUESCA', 'BURGOS', 'SORIA', 'CACERES', 'CIUDAD REAL', 'TOLEDO', 'SEVILLA', 'CORDOBA', 'MADRID', 'BARCELONA', 'VALENCIA'] 
provincias_filtradas = (filtrar('provincia', provincias_interes))
provincias_encontradas = set(provincia.get('provincia') for provincia in provincias_filtradas)
print(f"Se han encontrado estaciones en las siguientes provincias: {', '.join(sorted(provincias_encontradas))}")
print(f"Se han encontrado {len(provincias_filtradas)} estaciones en las provincias filtradas.")


In [ ]:
zonas=[
    "LA MUELA", "BORJA", "FUENDETODOS", "EPILA",
    "TARDIENTA",
    "NAVALMORAL", "ALCANTARA",
    "PUERTOLLANO",
    "TOLEDO", "BARGAS",
    "SEVILLA", "ALCALA DE GUADAIRA",
    "BURGOS",
    "SORIA",
    "MADRID", "BARAJAS",
    "BARCELONA", "EL PRAT",
    "VALENCIA"]
zonas_filtradas = filtrar('nombre', zonas)
print(f"Se han encontrado {len(zonas_filtradas)} estaciones en las zonas filtradas.")
nombres_solo = [estacion.get('nombre') for estacion in zonas_filtradas]
print(nombres_solo)

In [ ]:
def explorar_provincia(nombre_provincia):
    zonas = filtrar('provincia', [nombre_provincia])
    pueblos = [estacion.get('nombre') for estacion in zonas]
    print(f"Se han encontrado {len(pueblos)} estaciones en la provincia de {nombre_provincia}.")
    print(sorted(pueblos))

explorar_provincia('ZARAGOZA')
explorar_provincia('HUESCA')
explorar_provincia('CACERES')
explorar_provincia('CIUDAD REAL')
explorar_provincia('SEVILLA')
explorar_provincia('BURGOS')
explorar_provincia('SORIA')
explorar_provincia('BARCELONA')
explorar_provincia('MADRID')
explorar_provincia('VALENCIA')



In [ ]:
estaciones_interes = ['VALMADRID','ZARAGOZA, AEROPUERTO',
                      'ALMUDÉVAR','SARIÑENA',
                      'NAVALMORAL DE LA MATA','TRUJILLO',
                      'ALCAZAR DE SAN JUAN','PUERTOLLANO',
                      'CARMONA','ÉCIJA',
                      'MIRANDA DE EBRO','MEDINA DE POMAR',
                      'SAN PEDRO MANRIQUE',
                      'BARCELONA AEROPUERTO',
                      'MADRID AEROPUERTO',
                      'VALENCIA AEROPUERTO'
                      ]

estaciones_finales = filtrar('nombre', estaciones_interes)
print(f"Se han encontrado {len(estaciones_finales)} estaciones en la lista de interés.")
print(estaciones_finales)

In [ ]:
lista_estaciones_finales=[]
for elemento in estaciones_finales:
 if elemento.get('indicativo')  != '3463X':  
     lista_estaciones_finales.append(elemento)
 else:
    pass 
print(lista_estaciones_finales)

In [ ]:
lista_indicativos = []
for estacion in lista_estaciones_finales:
     lista_indicativos.append(estacion.get('indicativo'))
print(len(lista_indicativos))
print(lista_indicativos)


In [ ]:
lista_tramos = []
fecha_actual = date(2023, 1, 1)
fecha_limite = date(2026, 6, 30)

while fecha_actual <= fecha_limite:
    fin_tramo = fecha_actual + timedelta(days=182)  # aproximadamente 6 meses
    if fin_tramo > fecha_limite:
        fin_tramo = fecha_limite
    
    inicio_str = fecha_actual.strftime("%Y-%m-%dT00:00:00UTC")
    fin_str = fin_tramo.strftime("%Y-%m-%dT23:59:59UTC")
    
    lista_tramos.append((inicio_str, fin_str))
    
    fecha_actual = fin_tramo + timedelta(days=1)

print(len(lista_tramos))
print(lista_tramos)


In [ ]:
lista_resultados = []
load_dotenv(dotenv_path="../.env")
API_KEY = os.getenv("AEMET_API_KEY").strip().strip("'")
URL = f"https://opendata.aemet.es/opendata/api/valores/climatologicos/diarios/datos/fechaini/2023-01-01T00:00:00UTC/fechafin/2023-01-02T23:59:59UTC/estacion/0076"
headers = {
        "accept": "application/json",
        "api_key": API_KEY
    }

r = requests.get(URL, headers=headers)
print(r.status_code)
print(r.text)
datos_respuesta = r.json()
print(datos_respuesta)

data_url = datos_respuesta["datos"]
data_response = requests.get(data_url, headers=headers)
estaciones = data_response.json()
print(estaciones[:5])


In [ ]:

lista_resultados = []
lista_errores = []
load_dotenv(dotenv_path="../.env")
API_KEY = os.getenv("AEMET_API_KEY").strip().strip("'")

for indicativo in lista_indicativos:
    for inicio, fin in lista_tramos:
        URL = f"https://opendata.aemet.es/opendata/api/valores/climatologicos/diarios/datos/fechaini/{inicio}/fechafin/{fin}/estacion/{indicativo}"
        descarga_exitosa = False
        headers = {
            "accept": "application/json",
            "api_key": API_KEY
        }

        while True:
            if descarga_exitosa == True:
                break
            elif descarga_exitosa == False:
                try:
                    r = requests.get(URL, headers=headers)
                except requests.exceptions.ConnectionError:
                    print(f"fallo de conexión para {indicativo}, tramo {inicio}-{fin}")
                    time.sleep(5)
                else:
                    print(r.status_code)
                    print(r.text)
                    if r.status_code == 200:
                        datos_respuesta_2 = r.json()
                        print(datos_respuesta_2)
                        data_URL = datos_respuesta_2.get("datos")
                        estado_interno = datos_respuesta_2.get("estado")
                        if estado_interno == 200:
                            while True:
                                try:
                                    data_response = requests.get(data_URL, headers=headers)
                                    texto_guardado = data_response.text
                                    codigo_guardado = data_response.status_code
                                    print(data_response.status_code)
                                    print(data_response.text)
                                except requests.exceptions.ConnectionError:
                                    print(f"fallo de conexión para {indicativo}, tramo {inicio}-{fin}")
                                    time.sleep(5)
                                else:
                                    try:
                                        datos_climaticos = data_response.json()
                                        lista_resultados.append(datos_climaticos)
                                        print(f"descargado con exito ")
                                    except JSONDecodeError:
                                        print(f" la respuesta no era JSON para ese {indicativo}")
                                        descarga_exitosa = False 
                                        break
                                    print(lista_resultados[:5]) 
                                    descarga_exitosa = True
                                    break         
                        elif estado_interno != 200:
                            error_actual = {"indicativo": indicativo, "inicio":inicio,"fin":fin,"estado" :estado_interno}
                            lista_errores.append(error_actual)
                            break
                    elif r.status_code == 429:
                        print(f"esperar 60 segundos")
                        time.sleep(60)
                    elif r.status_code != 200:
                        print(f"Fallo en {indicativo}, tramo {inicio}-{fin}, código {r.status_code}")
                        break

In [ ]:
print(len(lista_resultados))
print(len(lista_errores))

In [ ]:
print(lista_resultados)

In [ ]:
with open(r"C:\Users\jaime\Desktop\Jaime_LLorca\proyectos\prediccion-electrica\data\raw\aemet\aemet_datos_raw.json", "w") as archivo:
    json.dump(lista_resultados, archivo)

In [ ]:
lista_plana=[]
for resultado in lista_resultados:
    for registro in resultado:
        lista_plana.append(registro)
print(lista_plana)

tabla_AEMT = pd.DataFrame(lista_plana)


### Nota: sustitución de estación 3463X por 3463Y (Trujillo, Cáceres)

Durante la verificación de las 7 combinaciones estación/tramo que en sesiones anteriores 
aparecían como "perdidas", se detectó que las 7 correspondían a una única estación: 
**3463X (Trujillo, Cáceres)**.

**Verificación:** se consultó la API de AEMET directamente para esta estación con distintos 
rangos de fechas, incluyendo tramos recientes (julio 2026), y en todos los casos se obtuvo 
un 404 constante. Esto descarta que el hueco se debiera a un error del bucle de descarga; 
la estación simplemente no reporta datos climatológicos diarios.

**Solución:** se localizó una segunda estación en Trujillo, **3463Y**, que sí devuelve datos 
(código 200 verificado). Se sustituyó 3463X por 3463Y en `estaciones_finales`.

**Resultado:** al relanzar el bucle con las 16 estaciones corregidas, se obtuvieron 
112/112 combinaciones exitosas, 0 errores.


In [ ]:
print(tabla_AEMT.shape)

In [ ]:
print(tabla_AEMT.head())
print(type(tabla_AEMT))
print(len(lista_plana))



In [ ]:
tabla_AEMT.groupby('indicativo').size()

In [ ]:

tabla_9051 = tabla_AEMT[tabla_AEMT['indicativo'] == '9051'].copy()


tabla_9051['fecha'] = pd.to_datetime(tabla_9051['fecha'])

rango_completo = pd.date_range(start='2023-01-01', end='2026-06-30', freq='1D')
fechas_presentes = set(tabla_9051['fecha'])
fechas_faltantes = [f for f in rango_completo if f not in fechas_presentes]

# 5. Inspeccionar
print(len(fechas_faltantes))  

In [ ]:
fechas_faltantes_ordenadas = sorted(fechas_faltantes)

# Mira las primeras y últimas para hacerte una idea rápida
print(fechas_faltantes_ordenadas[:5])
print(fechas_faltantes_ordenadas[-5:])

In [ ]:
diferencias = pd.Series(fechas_faltantes_ordenadas).diff()
diferencias.value_counts()

In [ ]:
# Índices donde la diferencia es mayor a 1 día (ahí empieza una racha nueva)
saltos = diferencias[diferencias > pd.Timedelta(days=1)].index

# Usamos esos índices para trocear fechas_faltantes_ordenadas en sub-listas
inicio_racha = 0
for salto in list(saltos) + [len(fechas_faltantes_ordenadas)]:
    racha = fechas_faltantes_ordenadas[inicio_racha:salto]
    print(f"Racha de {len(racha)} días: {racha[0]} → {racha[-1]}")
    inicio_racha = salto

### Nota: huecos temporales en estación 9051 (Medina de Pomar, Burgos)

Al verificar la cobertura de `tabla_AEMT` con `groupby('indicativo').size()`, la estación 
**9051 (Medina de Pomar, Burgos)** mostró el mayor déficit de todas: 88 días sin datos sobre 
un total teórico de 1.277 días (2023-01-01 a 2026-06-30).

**Análisis de distribución:** se comprobó que los 88 días faltantes no están dispersos 
aleatoriamente, sino agrupados en **4 rachas continuas**:

| Racha | Rango | Duración |
|---|---|---|
| 1 | 2023-01-18 → 2023-02-22 | 36 días |
| 2 | 2023-05-31 | 1 día |
| 3 | 2025-10-20 → 2025-10-27 | 8 días |
| 4 | 2026-04-17 → 2026-05-29 | 43 días |

**Búsqueda de causa:** se buscó si existía algún aviso o incidencia pública de AEMET que 
explicara estas paradas (avería, mantenimiento, sustitución de sensor). No se encontró 
documentación pública que lo confirme — AEMET no publica un registro accesible de este tipo 
de incidencias a nivel de estación individual.

**Interpretación:** el patrón (dos rachas largas de semanas y dos cortas) es compatible con 
mantenimiento o avería de sensor, más que con un error del pipeline de ingesta propio 
(ya verificado: la cuenta de 88 días faltantes coincide exactamente entre el `groupby` y el 
cálculo manual con `pd.date_range`).

**Pendiente / decisión futura:** definir estrategia de tratamiento de estos huecos en la fase 
de *feature engineering* (ej. interpolación para variables suaves como temperatura, exclusión 
o flag explícito para variables más erráticas como precipitación). No se resuelve en esta 
sesión — queda documentado para retomarlo cuando se aborde el preprocesado hacia `data/processed/`.


In [ ]:
tabla_AEMT.to_csv(r"C:\Users\jaime\Desktop\Jaime_LLorca\proyectos\prediccion-electrica\data\interim\tabla_AEMT", index=False, encoding="utf-8")